# Practice 4-2. 다중 질의 생성 (MultiQuery Generation)

원본 질문을 LLM으로 여러 관점의 하위 질의로 변형한 뒤, 각 질의로 독립적으로 검색하고 결과를 통합하는 질의 변형 기법입니다.
책은 OpenAI(gpt-4o)를 사용하지만, 이 노트북은 LM Studio 로컬 LLM과 Chroma 서버로 동일한 파이프라인을 구성합니다.

## 0. 환경 설정

In [ ]:
import re
import chromadb

# --- LM Studio ---
LMSTUDIO_BASE_URL = "http://...:.../v1"         # /v1 까지 포함
LMSTUDIO_API_KEY  = "lm-studio"                              # LM Studio는 키 검증 안 함 (더미)

EMBED_MODEL = "text-embedding-qwen3-embedding-8b"
LLM_MODEL   = "qwen3-30b-a3b-instruct-2507"   # LM Studio에 로드되어 있는 로컬 채팅 모델

# --- Chroma 서버 ---
CHROMA_HOST = "chromadb"
CHROMA_PORT = 8000
COLLECTION  = "practice4_multiquery"

# --- 입력 파일 (노트북 기준 상대경로) ---
DATA_PATH = "How_to_invest_money.txt"


def strip_think(text: str) -> str:
    """추론 모델의 <think>...</think> 블록 제거."""
    if text is None:
        return ""
    return re.sub(r"(^|<think>).*?</think>", "", text, flags=re.DOTALL).strip()


chroma_client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
print("Chroma 서버 연결:", chroma_client.heartbeat())

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# 임베딩 모델
# check_embedding_ctx_length=False : tiktoken 대신 원문 문자열을 전송 → LM Studio 호환
# chunk_size=8 : 로컬 LM Studio 서버가 한 번에 받는 텍스트 수가 많으면 연결이 끊기므로 작게 배치
embeddings = OpenAIEmbeddings(
    model=EMBED_MODEL,
    base_url=LMSTUDIO_BASE_URL,
    api_key=LMSTUDIO_API_KEY,
    check_embedding_ctx_length=False,
    chunk_size=8,
)

# 채팅 모델 (질의 변형 + 최종 답변 생성용)
# LM Studio의 로컬 모델은 추론(사고) 모델이라 답변 전에 reasoning 토큰을 소모하므로
# temperature=0, max_tokens는 넉넉히 잡아 답변이 잘리지 않도록 한다.
llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LMSTUDIO_BASE_URL,
    api_key=LMSTUDIO_API_KEY,
    temperature=0,
    max_tokens=2048,
)

print("EMBED:", len(embeddings.embed_query("연결 테스트")), "차원")
print("LLM  :", strip_think(llm.invoke("연결 테스트. 'ok' 라고만 답하세요.").content)[:50])

## 1. 문서 로드 · 분할 · Chroma 벡터스토어 구축

`How_to_invest_money.txt`를 로드한 뒤 1000자 단위(중복 200자)로 분할하고, Chroma 서버에 임베딩하여 저장합니다.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# 문서 로더 설정 (encoding="utf-8-sig" : 원본 파일 선두의 BOM 제거)
loaders = [TextLoader(DATA_PATH, encoding="utf-8-sig")]

docs = []
for loader in loaders:
    docs.extend(loader.load())

# 문서 생성을 위한 텍스트 분할기 정의
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# 문서 분할
split_docs = recursive_splitter.split_documents(docs)
print("분할된 문서 수:", len(split_docs))

# 재실행 안전: 이전 실행에서 남은 컬렉션 제거 후 새로 생성
try:
    chroma_client.delete_collection(COLLECTION)
except Exception:
    pass

# Chroma vectorstore 생성 (Chroma 서버 컬렉션에 임베딩하여 저장)
vectorstore = Chroma(client=chroma_client, collection_name=COLLECTION, embedding_function=embeddings)
vectorstore.add_documents(split_docs)
print("벡터스토어 저장 완료:", vectorstore._collection.count(), "개")

## 2. MultiQueryRetriever 구성

로깅을 설정해 LLM이 생성한 하위 질의들을 콘솔에서 확인할 수 있게 합니다. `langchain` 1.x 에서 `MultiQueryRetriever` 의 위치가 `langchain_classic` 으로 이동했으므로 폴백 임포트를 사용합니다.

In [ ]:
# 쿼리를 위한 로그 설정 (생성된 하위 질의들을 콘솔에서 확인)
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

# MultiQueryRetriever: langchain 1.x 에서 위치 이동 → 폴백 처리
try:
    from langchain_classic.retrievers.multi_query import MultiQueryRetriever
except ImportError:
    from langchain.retrievers import MultiQueryRetriever

# MultiQueryRetriever 실행
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),  # 기본 검색기(벡터 데이터베이스)
    llm=llm,                                # 앞서 정의한 로컬 llm
)

print("retriever 준비 완료:", type(retriever).__name__)

## 3. 검색 확인

"주식 투자를 처음 시작하려면 어떻게 해야 하나요?"라고 질문해 보겠습니다. 질문은 `invoke()` 메서드를 통해 retriever 객체에 전달되어 여러 개의 다른 질문으로 변형된 뒤 벡터 데이터베이스에 전달됩니다.

In [ ]:
# 예시 질문
question = "주식 투자를 처음 시작하려면 어떻게 해야 하나요?"

# 결과 검색
unique_docs = retriever.invoke(question)
print(f"\n결과: {len(unique_docs)}개의 문서가 검색되었습니다.")

In [ ]:
unique_docs

## 4. 답변 생성 (RetrievalQA)

검색된 문서들을 활용하여 최종 답변을 생성하는 `RetrievalQA` 체인을 설정합니다. `langchain_classic` 폴백 임포트는 앞서와 동일합니다.

In [ ]:
# RetrievalQA: langchain 1.x 에서 위치 이동 → 폴백 처리
try:
    from langchain_classic.chains import RetrievalQA
except ImportError:
    from langchain.chains import RetrievalQA

# RetrievalQA 체인 설정
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)

# 질문에 대한 답변 생성
result = qa_chain.invoke({"query": question})

# 결과 출력
print("답변:", strip_think(result["result"]))
print("\n사용된 문서 수:", len(result["source_documents"]))